# Practical 11 — RegTech, Compliance & AML Screening

This practical screens a set of bank transactions for money-laundering red
flags and drafts a SAR-style narrative (Suspicious Activity Report) that cites
the specific rule behind every claim. All detection is done by **transparent,
deterministic rules** in `tools/screen.py` — structuring, round amounts,
high-risk jurisdictions, and velocity — so no model ever decides on its own
that an amount, date window, or country is suspicious.

In Claude Code / Cline this is an *agentic* project where the agent chooses
which rules to run and writes the narrative. This notebook is the Colab-friendly
counterpart: it drives the same reference tools directly, end to end, fully
offline, so you can watch the whole screen -> group -> draft -> review pipeline
run on the real bundled data.

In [ ]:
# This practical's tools are Python standard library only —
# nothing to install. (Colab already has Python ready to go.)

In [ ]:
# --- setup: make this practical's tools importable (works locally & in Colab) ---
import os, sys, pathlib, subprocess
PRACTICAL = "11-regtech-compliance-aml"
def _locate():
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if base.name == PRACTICAL and (base / "tools").is_dir():
            return base
        cand = base / "code" / "practicals" / PRACTICAL
        if cand.is_dir():
            return cand
    return None
root = _locate()
if root is None:
    if not pathlib.Path("llm-finance-book").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/jfimbett/llm-finance-book.git"], check=True)
    root = pathlib.Path("llm-finance-book/code/practicals") / PRACTICAL
root = root.resolve()
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "tools"))
os.chdir(root)
print("Practical root:", root)

## Step 0 — Load the bundled data

The data is two small, fictional, offline files:

- `data/transactions.csv` — columns `id, date, amount, origin_country,
  dest_country, account`.
- `data/high_risk.json` — a list of high-risk ISO country codes (fictional,
  *not* a real FATF/sanctions list).

The helpers in `tools/_common.py` parse these into typed `Transaction` records
and a set of high-risk country codes.

In [ ]:
import json
from tools._common import load_transactions, load_high_risk

txns = load_transactions()
high_risk = load_high_risk()

print(f"Loaded {len(txns)} transactions across "
      f"{len({t.account for t in txns})} accounts.")
print("High-risk jurisdictions:", sorted(high_risk))
print()
print(f"{'id':<6}{'date':<18}{'amount':>12}  {'orig->dest':<12}{'account'}")
for t in txns:
    print(f"{t.id:<6}{t.dt:%Y-%m-%d %H:%M}  {t.amount:>10,.2f}  "
          f"{t.origin_country+'->'+t.dest_country:<12}{t.account}")

## Step 1 — Screen: run the AML rules

`tools/screen.py` defines four pure-function rules with explicit, named
thresholds (no hidden state):

| Rule | Fires when |
|------|-----------|
| `structuring` | >= 3 deposits in `[9,000, 10,000)` on one account within 7 days |
| `round_number` | amount is an exact multiple of 1,000 and >= 5,000 |
| `high_risk_jurisdiction` | origin or destination is on the high-risk list |
| `velocity` | >= 4 transactions on one account within 24 hours |

Each rule returns a `{transaction_id: reason}` map; the `reason` string names
the exact figure, window, or country that tripped it, so the narrative can
cite the flag instead of inventing one. We can run a single rule first.

In [ ]:
from tools.screen import (
    rule_structuring, rule_round_number,
    rule_high_risk_jurisdiction, rule_velocity, screen,
)

print("structuring rule, run alone:")
struct = rule_structuring(txns)
print(json.dumps(struct, indent=2))

Now run **all** rules together with `screen()`. It returns
`{transaction_id: {account, date, amount, flags: {rule: reason}}}`, ordered by
date then id. A single transaction can trip more than one rule.

In [ ]:
flags = screen()
print(f"{len(flags)} transaction(s) flagged.\n")
print(json.dumps(flags, indent=2))

This is exactly what `python -m tools.screen --json > reports/_flags.json`
produces. Let's persist it the same way the agent does, so the later steps
read from the saved artifact.

In [ ]:
flags_path = root / "reports" / "_flags.json"
flags_path.parent.mkdir(exist_ok=True)
flags_path.write_text(json.dumps(flags, indent=2), encoding="utf-8")
print("Wrote", flags_path)

### Try it / interpret

The README promises one rule per "bad" account. Let's confirm which rule
fired on which account, and that the benign accounts `ACC100` and `ACC105`
produce **no** flags (no false positives).

In [ ]:
by_rule = {}
for tid, info in flags.items():
    for rule in info["flags"]:
        by_rule.setdefault(rule, set()).add(tid)

for rule, ids in sorted(by_rule.items()):
    accts = sorted({flags[t]["account"] for t in ids})
    print(f"{rule:<24} -> {sorted(ids)}  (accounts {accts})")

flagged_accounts = {info["account"] for info in flags.values()}
print("\nBenign ACC100 flagged?", "ACC100" in flagged_accounts)
print("Benign ACC105 flagged?", "ACC105" in flagged_accounts)

As promised: structuring on `ACC201`, round-number on `ACC202`, high-risk on
`ACC203`, velocity on `ACC204`; the two benign accounts stay clean.

## Step 2 — Group the flags by account

The SAR narrative is written one section per account, so we regroup the flat
flag map into `{account: [flag entries]}`. This mirrors the `screener` /
`sar-writer` hand-off in the agentic version (which reads `reports/_flags.json`).

In [ ]:
saved_flags = json.loads(flags_path.read_text(encoding="utf-8"))

by_account = {}
for tid, info in saved_flags.items():
    by_account.setdefault(info["account"], []).append({"id": tid, **info})

for account, entries in sorted(by_account.items()):
    rules_here = sorted({r for e in entries for r in e["flags"]})
    print(f"{account}: {len(entries)} flagged txn(s), rules {rules_here}")

## Step 3 — Draft a grounded SAR narrative

A SAR narrative makes one section per account and, after every assertion,
cites the transaction ids and the rule that supports it, e.g.
`(T010, T011, T012 — structuring)`. Crucially, every sentence is generated
*from a tool flag* — nothing is invented. Here we build that narrative
deterministically straight from the saved flags.

In [ ]:
from datetime import date

def draft_sar(by_account: dict) -> str:
    lines = ["# Suspicious Activity Report (draft)",
             f"_Generated {date.today():%Y-%m-%d} from deterministic AML rules._", ""]
    if not by_account:
        lines.append("No suspicious activity detected by the configured rules.")
        return "\n".join(lines)
    for account in sorted(by_account):
        entries = by_account[account]
        lines.append(f"## Account {account}")
        # one bullet per (rule) grouping the transactions that tripped it
        rule_to_ids = {}
        rule_to_reason = {}
        for e in entries:
            for rule, reason in e["flags"].items():
                rule_to_ids.setdefault(rule, []).append(e["id"])
                rule_to_reason[rule] = reason
        for rule in sorted(rule_to_ids):
            ids = ", ".join(sorted(rule_to_ids[rule]))
            lines.append(f"- {rule_to_reason[rule]} ({ids} — {rule}).")
        lines.append("")
    return "\n".join(lines)

narrative = draft_sar(by_account)
print(narrative)

## Step 4 — Review: every claim must trace to a flag

The `reviewer` step is the safety net: it confirms every citation in the
narrative points at a transaction that actually fired that rule in
`reports/_flags.json`, and that no flag was silently dropped. We verify both
directions here.

In [ ]:
import re

cited = set()  # (transaction_id, rule) pairs claimed in the narrative
for m in re.finditer(r"\(([^()]*?)\s+—\s+([a-z_]+)\)", narrative):
    ids, rule = m.group(1), m.group(2)
    for tid in re.split(r",\s*", ids.strip()):
        cited.add((tid, rule))

actual = {(tid, rule) for tid, info in saved_flags.items() for rule in info["flags"]}

unsupported = cited - actual   # claims with no backing flag -> must be rejected
dropped = actual - cited       # flags the narrative forgot to mention

print(f"Claims in narrative : {len(cited)}")
print(f"Flags in tool output: {len(actual)}")
print("Unsupported claims  :", sorted(unsupported) or "none")
print("Dropped flags       :", sorted(dropped) or "none")
assert not unsupported, "narrative cites a flag the tools never produced"
assert not dropped, "narrative dropped a flag the tools produced"
print("\nReview passed: every claim is grounded and no flag was dropped.")

### Try it / interpret — reject an unsupported claim

The README suggests asking the agent to assert a transaction the tool did
**not** flag; the reviewer should reject it. Let's simulate that: append a
bogus citation for `T001` (a benign transaction) and re-run the check.

In [ ]:
tampered = narrative + "\n- fabricated suspicion (T001 — structuring).\n"
bad_cited = set()
for m in re.finditer(r"\(([^()]*?)\s+—\s+([a-z_]+)\)", tampered):
    for tid in re.split(r",\s*", m.group(1).strip()):
        bad_cited.add((tid, m.group(2)))
rejected = bad_cited - actual
print("Reviewer would reject these unsupported claims:", sorted(rejected))
assert ("T001", "structuring") in rejected
print("Correctly rejected the fabricated T001 claim.")

## Step 5 — Save the report

The agentic version saves the narrative (and the flags) to
`reports/sar_<date>.md`. We do the same here.

In [ ]:
report_path = root / "reports" / f"sar_{date.today():%Y-%m-%d}.md"
report_path.write_text(narrative, encoding="utf-8")
print("Saved SAR report ->", report_path)
print(f"({len(narrative.splitlines())} lines, {len(by_account)} account sections)")

## Recap / next steps

We ran the full AML pipeline offline on the bundled data:

1. **Load** the fictional transactions and high-risk jurisdiction list.
2. **Screen** with four transparent rules (`structuring`, `round_number`,
   `high_risk_jurisdiction`, `velocity`) and saved `reports/_flags.json`.
3. **Group** the flags by account.
4. **Draft** a SAR narrative whose every claim cites a tool flag.
5. **Review** that every claim is grounded and no flag was dropped, and saw
   the reviewer reject a fabricated claim about a benign transaction.

Things to explore next (edit the files, then re-run this notebook):

- Lower `STRUCTURING_BAND_LOW` in `tools/screen.py` to `8,000` and watch clean
  rows start tripping (and a pytest turn red: `python -m pytest -q`).
- Add a country to `data/high_risk.json` that a benign transaction routes
  through (e.g. `CA` or `GB`) and re-screen to see a new flag appear.
- Add a fifth account to `data/transactions.csv` that trips two rules at once
  and confirm the narrative cites both.

The key RegTech lesson: detection stays in auditable, testable rules; the LLM
only orchestrates and explains — and the reviewer step keeps it honest.